In [1]:
import tensorflow as tf
from tensorflow.keras.datasets import imdb
from tensorflow.keras.preprocessing.sequence import pad_sequences
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import GRU, Embedding, Dense

# Parameters
max_features = 10000  # Top 10,000 words
maxlen = 500  # Maximum length of reviews (in terms of words)
embedding_size = 128

# Load IMDb dataset
(x_train, y_train), (x_test, y_test) = imdb.load_data(num_words=max_features)

17464789/17464789 ━━━━━━━━━━━━━━━━━━━━ 0s 0us/step


In [2]:
# Pad sequences to the same length
x_train = pad_sequences(x_train, maxlen=maxlen)
x_test = pad_sequences(x_test, maxlen=maxlen)

In [3]:
# Build a GRU model
model_gru = Sequential()
model_gru.add(Embedding(max_features, embedding_size, input_length=maxlen))
model_gru.add(GRU(64))  # GRU layer with 64 units
model_gru.add(Dense(1, activation='sigmoid'))  # Output layer for binary classification

/usr/local/lib/python3.12/dist-packages/keras/src/layers/core/embedding.py:100: UserWarning: Argument `input_length` is deprecated. Just remove it.
  warnings.warn(


In [4]:
# Compile model
model_gru.compile(optimizer='adam', loss='binary_crossentropy', metrics=['accuracy'])

# Train the model
model_gru.fit(x_train, y_train, epochs=3, batch_size=64, validation_data=(x_test, y_test))

Epoch 1/3
391/391 ━━━━━━━━━━━━━━━━━━━━ 349s 886ms/step - accuracy: 0.7994 - loss: 0.4216 - val_accuracy: 0.8625 - val_loss: 0.3270
Epoch 2/3
391/391 ━━━━━━━━━━━━━━━━━━━━ 327s 836ms/step - accuracy: 0.9023 - loss: 0.2494 - val_accuracy: 0.8811 - val_loss: 0.2984
Epoch 3/3
391/391 ━━━━━━━━━━━━━━━━━━━━ 326s 834ms/step - accuracy: 0.9351 - loss: 0.1748 - val_accuracy: 0.8816 - val_loss: 0.3077


In [5]:
# Evaluate the model
loss, accuracy = model_gru.evaluate(x_test, y_test)
print(f"GRU Model Test Accuracy: {accuracy}")

782/782 ━━━━━━━━━━━━━━━━━━━━ 60s 77ms/step - accuracy: 0.8816 - loss: 0.3077
GRU Model Test Accuracy: 0.881600022315979


In [6]:
# Sample texts for real-time testing
texts = [
  "I absolutely loved this movie, the storyline was engaging!",
  "The film was too slow and boring for my taste.",
  "A perfect blend of comedy and drama, truly enjoyable!"
]

In [7]:
# Load IMDb word index
word_index = tf.keras.datasets.imdb.get_word_index()

# Tokenizer for IMDb dataset
tokenizer = tf.keras.preprocessing.text.Tokenizer(num_words=max_features)
tokenizer.fit_on_texts([' '.join([str(word) for word in sequence]) for sequence in x_train])

# Convert texts to sequences
sequences = tokenizer.texts_to_sequences(texts)

# Pad the sequences
padded_sequences = pad_sequences(sequences, maxlen=maxlen)

1641221/1641221 ━━━━━━━━━━━━━━━━━━━━ 0s 0us/step


In [8]:
# Predict sentiment
predictions = model_gru.predict(padded_sequences)

# Output the results
for i, text in enumerate(texts):
    sentiment = "Positive" if predictions[i] > 0.5 else "Negative"
    print(f"Text: {text}\nPredicted Sentiment: {sentiment}\n")

1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 251ms/step
Text: I absolutely loved this movie, the storyline was engaging!
Predicted Sentiment: Positive

Text: The film was too slow and boring for my taste.
Predicted Sentiment: Positive

Text: A perfect blend of comedy and drama, truly enjoyable!
Predicted Sentiment: Positive



In [ ]:
The movie was great

In [ ]:
Time step t: Processing word "great"

Input: x_t = embedding vector for "great" (size: 128)
Hidden state from previous step: h_{t-1} (size: 64)

INTERNAL CALCULATIONS:

1. UPDATE GATE (z_t) - "How much to update the hidden state"
   z_t = σ(W_z · x_t + U_z · h_{t-1} + b_z)

   Example numbers:
   W_z · x_t = [0.2, -0.5, 0.8, ...] dot product = 0.45
   U_z · h_{t-1} = [0.3, 0.1, -0.2, ...] dot product = 0.23
   z_t = sigmoid(0.45 + 0.23 + 0.1) = sigmoid(0.78) = 0.686

2. RESET GATE (r_t) - "How much to forget previous state"
   r_t = σ(W_r · x_t + U_r · h_{t-1} + b_r)

   r_t = sigmoid(0.35 + 0.18 + 0.05) = sigmoid(0.58) = 0.641

3. CANDIDATE HIDDEN STATE (h̃_t) - "New candidate memory"
   h̃_t = tanh(W_h · x_t + U_h · (r_t ⊙ h_{t-1}) + b_h)

   r_t ⊙ h_{t-1} = 0.641 * [0.5, -0.3, 0.2, ...] = [0.32, -0.19, 0.13, ...]
   h̃_t = tanh(0.52 + 0.41 + 0.08) = tanh(1.01) = 0.77

4. FINAL HIDDEN STATE (h_t) - "Combine old and new"
   h_t = (1 - z_t) ⊙ h_{t-1} + z_t ⊙ h̃_t

   = (1 - 0.686) * [0.5, -0.3, 0.2] + 0.686 * [0.6, -0.2, 0.4]
   = 0.314 * [0.5, -0.3, 0.2] + 0.686 * [0.6, -0.2, 0.4]
   = [0.157, -0.094, 0.063] + [0.412, -0.137, 0.274]
   = [0.569, -0.231, 0.337]

In [ ]:
max_features = 10000  # Top 10,000 words
maxlen = 500          # Maximum review length
embedding_size = 128

(x_train, y_train), (x_test, y_test) = imdb.load_data(num_words=max_features)

In [ ]:
# The IMDB dataset has already converted words to integers
# Example review: "the movie was great"
# Becomes: [1, 14, 22, 16]  (indices in vocabulary)

# Vocabulary mapping (partial):
word_index = {
    'the': 1,
    'a': 2,
    'movie': 14,
    'film': 15,
    'was': 22,
    'great': 16,
    'good': 27,
    'bad': 38,
    ...
}

In [ ]:
x_train.shape  # (25000, variable_length) - each review has different length
y_train.shape  # (25000,) - 0 for negative, 1 for positive

# Sample from training data:
print(x_train[0])  # [1, 14, 22, 16, 42, 8, 3, 127, ...]
print(y_train[0])  # 1 (positive sentiment)

In [ ]:
x_train = pad_sequences(x_train, maxlen=maxlen)
x_test = pad_sequences(x_test, maxlen=maxlen)

In [ ]:
# Original review with 3 words: [1, 14, 22]
# After padding to maxlen=500:
padded_review = [1, 14, 22, 0, 0, 0, ..., 0]
#                ^^^ original ^^^  ^^^^^ padding zeros ^^^^^

# Shape becomes: (25000, 500) - every review now has exactly 500 integers

In [ ]:
model_gru = Sequential()
model_gru.add(Embedding(max_features, embedding_size, input_length=maxlen))
model_gru.add(GRU(64))
model_gru.add(Dense(1, activation='sigmoid'))

In [ ]:
Input: Integer sequence of length 500
[1, 14, 22, 16, 0, 0, ...]

Embedding matrix (10000 × 128):
Row 0: [0.01, -0.02, 0.03, ...]  # Padding token
Row 1: [0.12, 0.34, -0.56, ...]  # 'the'
Row 2: [0.23, -0.45, 0.67, ...]  # 'a'
...
Row 14: [-0.11, 0.78, 0.23, ...] # 'movie'
Row 16: [0.89, -0.34, 0.12, ...] # 'great'

Output shape: (batch_size, 500, 128)
Each integer becomes a 128-dimensional vector

In [ ]:
# Input batch (2 reviews, each 500 tokens)
batch = [[1, 14, 22, 16, 0, ...],
         [38, 1, 42, 22, 0, ...]]

# Embedding lookup (simplified)
output[0, 0] = embedding_matrix[1]  # 'the' vector
output[0, 1] = embedding_matrix[14] # 'movie' vector
output[0, 2] = embedding_matrix[22] # 'was' vector
output[0, 3] = embedding_matrix[16] # 'great' vector
output[0, 4] = embedding_matrix[0]  # padding vector

In [ ]:
# Each GRU cell has 3 weight matrices and 3 bias vectors
# For input size 128 → hidden size 64:

W_z shape: (128, 64)  # Update gate input weights
U_z shape: (64, 64)   # Update gate recurrent weights
b_z shape: (64,)      # Update gate bias

W_r shape: (128, 64)  # Reset gate input weights
U_r shape: (64, 64)   # Reset gate recurrent weights
b_r shape: (64,)      # Reset gate bias

W_h shape: (128, 64)  # Candidate hidden input weights
U_h shape: (64, 64)   # Candidate hidden recurrent weights
b_h shape: (64,)      # Candidate hidden bias

# Total parameters: 3 × (128×64 + 64×64 + 64) = 3 × (8192 + 4096 + 64)
# = 3 × 12352 = 37,056 parameters for GRU layer